# Celda 0: Preparar el entorno

In [12]:
# recarga los .py de src/ al editarlos, sin reiniciar el kernel
%reload_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Las rutas son relativas, así que solo funcionan desde notebooks/
if not Path("../src/differential_expression.py").exists():
    raise FileNotFoundError(
        "No se encontró ../src/differential_expression.py. Solución: abre "
        "este notebook desde la carpeta notebooks/ del proyecto."
    )

sys.path.insert(0, "../src")

PROC = Path("../data/processed")
TABLES = Path("../results/tables")
FIGURES = Path("../results/figures")
for folder in (TABLES, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)

print("Python", sys.version.split()[0])
print("Entorno listo")

Python 3.13.9
Entorno listo


## Celda 1: cargar los datos de entrada
A partir del notebook 01_descarga.ipynb carga: los conteos crudos ya filtrados, los
metadatos con la condición de cada muestra y la anotación funcional del
genoma.

In [13]:
counts = pd.read_csv(PROC / "counts_filtered.csv", index_col=0)
meta = pd.read_csv(PROC / "sample_metadata.csv", index_col=0)
annotation = pd.read_csv(PROC / "final_annotation.csv")

print(f"Conteos    : {counts.shape[0]} genes × {counts.shape[1]} muestras")
print(f"Anotación  : {len(annotation)} genes")
print()
print(meta["condition"].value_counts().to_string())

Conteos    : 5394 genes × 15 muestras
Anotación  : 5394 genes

condition
plant      13
culture     2


## Celda 2: construir el objeto del modelo

Se transpone la matriz guardada como genes × muestras por el formato de 
InMoose que espera muestras × genes.


También designa el nivel de referencia como `culture`.

Se valida que los conteos sean enteros, necesario para
que el modelo binomial negativo no sería aplicable.

In [14]:
import differential_expression as de

dds = de.build_dataset(counts, meta, design="~condicion", reference="culture")

print("Muestras × genes:", dds.shape)
print("Diseño          :", dds.design)

Muestras × genes: (15, 5394)
Diseño          : [[1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 0.]
 [1. 0.]
 [1. 1.]
 [1. 1.]]


## Celda 3: normalización

Por el método de mediana de razones: para cada gen se calcula la media geométrica
a lo largo de las muestras como referencia; el factor de tamaño de una muestra
es la mediana de sus razones contra esa referencia.


In [15]:
dds, size_factors = de.normalize(dds)

print(size_factors.round(3).to_string())
print("*"*52)
print(f"Rango: {size_factors.min():.3f} – {size_factors.max():.3f}")

# comprobacion: se debe mostrar 
median_raw = counts.median()
median_norm = (counts / size_factors).median()
print("*"*52)
print(f"Mediana de conteos crudos       : {median_raw.min():.0f} – {median_raw.max():.0f}")
print(f"Mediana de conteos normalizados : {median_norm.min():.0f} – {median_norm.max():.0f}")

sample
Pto_col_A13    1.105
Pto_col_A14    1.826
Pto_col_A15    0.884
Pto_col_A17    1.421
Pto_col_A25    0.992
Pto_col_A26    1.696
Pto_col_A42    0.399
Pto_col_A66    0.687
Pto_col_A57    0.907
Pto_col_A58    1.280
Pto_col_A59    1.662
Pto_KB_A81     1.419
Pto_KB_A83     0.911
Pto_col_A71    0.678
Pto_col_A74    0.757
****************************************************
Rango: 0.399 – 1.826
****************************************************
Mediana de conteos crudos       : 28 – 128
Mediana de conteos normalizados : 67 – 77


## Celda 4: estimación de la dispersión

El parámetro de dispersión mide la variabilidad biológica entre réplicas, más
allá del ruido de muestreo. Con dos réplicas en King's B no es estimable gen
por gen, así que InMoose ajusta una tendencia usando los 5394 genes y acerca
cada estimación individual a esa curva.

La columna `dispGeneEst` es la estimación con un solo gen y `dispersion` la
final, ya encogida.

In [16]:
dds, dispersions = de.estimate_dispersions(dds)

resumen = dispersions.describe().loc[["min", "50%", "max"]].round(4)
print(resumen.to_string())

imposibles = int((dispersions["dispGeneEst"] <= 1e-8).sum())
print("*"*52)
print(f"Estimaciones individuales pegadas al límite inferior: {imposibles}")
print(f"Las mismas tras el encogimiento: "
      f"{int((dispersions['dispersion'] <= 1e-8).sum())}")

       baseMean  dispGeneEst  dispFit  dispersion
min      0.2880       0.0000   0.0539      0.0147
50%     76.3916       0.0928   0.0983      0.0986
max  66838.3114      15.0000  11.8517     15.0000
****************************************************
Estimaciones individuales pegadas al límite inferior: 34
Las mismas tras el encogimiento: 0


## Celda 5: ajustar el modelo y aplicar la prueba de Wald

Se ajusta un modelo lineal generalizado binomial negativo. El coeficiente de
la condición es directamente el log2 fold change, y la prueba de Wald evalúa
si difiere de cero.

In [17]:
dds = de.fit_model(dds)

print("Coeficientes del modelo:", dds.resultsNames())

Coeficientes del modelo: Index(['Intercept', 'condicion_plant_vs_culture'], dtype='object')


## Celda 6: extraer la tabla de resultados

El contraste compara `plant` contra `culture`. Los p-valores se corrigen por
pruebas múltiples con Benjamini-Hochberg; en InMoose esa columna se llama
`adj_pvalue`, no `padj`.

Los `NaN` en `adj_pvalue` no son un error: son genes de expresión muy baja que
el filtrado independiente descartó para recuperar poder estadístico.

In [18]:
results = de.extract_results(
    dds, counts.index, contrast=["condicion", "plant", "culture"], alpha=0.05
)

print(results.shape)
print("*"*75)
print(results.head().round(4).to_string())
print("*"*75)
print(f"Genes sin adj_pvalue (filtrado independiente): "
      f"{int(results['adj_pvalue'].isna().sum())}")

(5394, 6)
***************************************************************************
            baseMean  log2FoldChange   lfcSE    stat  pvalue  adj_pvalue
locus_key                                                               
PSPTO0001  2783.0514          1.0105  0.3228  3.1305  0.0017      0.0061
PSPTO0002   927.2627         -0.2348  0.2474 -0.9492  0.3425      0.4806
PSPTO0003   579.5052          1.3285  0.2382  5.5782  0.0000      0.0000
PSPTO0004  1997.4941          2.0171  0.2452  8.2273  0.0000      0.0000
PSPTO0005   113.3101          0.0063  0.3010  0.0208  0.9834      0.9880
***************************************************************************
Genes sin adj_pvalue (filtrado independiente): 427


## Celda 7: validación biológica

Antes de interpretar nada, se comprueba que los genes de referencia se
comporten como la literatura predice. *hrpL*, *avrPto*, *cmaA* y *pvdS*
pertenecen al programa de virulencia y deben inducirse durante la infección;
*gapA* y *gyrA* son constitutivos y deben mantenerse estables.

Si esto falla, el problema está en el diseño o en los datos, no en la
interpretación.

In [19]:
controles = de.validate_controls(results)
print(controles.to_string(index=False))
print("*"*52)
if (controles["cumple"] == "NO").any():
    print("\nATENCIÓN: algún control no se comporta como se esperaba.")
else:
    print("\nTodos los controles se comportan como se esperaba.")

   gen     tipo  log2FoldChange    adj_pvalue cumple
  hrpL inducido            4.90  2.297136e-61     sí
avrPto inducido            6.05  4.183819e-96     sí
  cmaA inducido            5.68 4.913921e-180     sí
  pvdS inducido            6.86  1.157969e-68     sí
  gapA  estable            1.44  6.561594e-09     sí
  gyrA  estable            0.99  1.793921e-07     sí
****************************************************

Todos los controles se comportan como se esperaba.


## Celda 8: clasificar y anotar

Un gen se considera diferencialmente expresado si su FDR es menor a 0.05 y su
cambio supera el doble en cualquier dirección (|log2FoldChange| > 1).

Después se une la tabla con la anotación funcional, que es lo que convierte
una lista de identificadores en un resultado interpretable.

In [20]:
ALPHA = 0.05
LFC_MIN = 1.0

results = de.classify_degs(results, alpha=ALPHA, lfc_min=LFC_MIN)
deg_table = de.annotate_results(results, annotation, key="locus_key")

print(deg_table["regulacion"].value_counts().to_string())
print("*" * 52)
print(f"Genes con anotación: "
      f"{int(deg_table['final_product'].notna().sum())} de {len(deg_table)}")

regulacion
sin cambio    3936
inducido       830
reprimido      628
****************************************************
Genes con anotación: 5394 de 5394


## Celda 9: guardar resultados

Se guardan dos tablas: la completa, con los 5394 genes, y la de los
diferencialmente expresados. La completa es la que hace falta para el volcano
plot, porque necesita también los genes sin cambio.

In [21]:
# los DEGs son el subconjunto significativo de la tabla completa
degs = deg_table[deg_table["regulacion"] != "sin cambio"]

deg_table.to_csv(TABLES / "deg_results_completa.csv")
degs.to_csv(TABLES / "deg_results_significativos.csv")

size_factors.to_csv(TABLES / "factores_de_tamano.csv")
controles.to_csv(TABLES / "validacion_controles.csv", index=False)
print("Archivos generados en results/tables:")
for archivo in sorted(TABLES.glob("*.csv")):
    print(f"{archivo.name:38s} {archivo.stat().st_size / 1024:7.1f} KB")

Archivos generados en results/tables:
deg_results_completa.csv                 974.8 KB
deg_results_significativos.csv           267.8 KB
factores_de_tamano.csv                     0.5 KB
validacion_controles.csv                   0.3 KB


# Celda 10: Resumen del análisis

In [22]:
n_up = int((deg_table["regulacion"] == "inducido").sum())
n_down = int((deg_table["regulacion"] == "reprimido").sum())

print("ANÁLISIS DE EXPRESIÓN DIFERENCIAL")
print("*" * 52)
print(f"Comparación          : in planta vs King's B")
print(f"Muestras             : {int((meta['condition'] == 'plant').sum())} in planta, "
      f"{int((meta['condition'] == 'culture').sum())} en cultivo")
print(f"Genes evaluados      : {len(deg_table)}")
print(f"Criterio             : FDR < {ALPHA} y |log2FC| > {LFC_MIN}")
print("*" * 52)
print(f"Inducidos in planta  : {n_up}")
print(f"Reprimidos in planta : {n_down}")
print(f"Total diferenciales  : {n_up + n_down} "
      f"({(n_up + n_down) / len(deg_table) * 100:.1f} % del genoma filtrado)")

ANÁLISIS DE EXPRESIÓN DIFERENCIAL
****************************************************
Comparación          : in planta vs King's B
Muestras             : 13 in planta, 2 en cultivo
Genes evaluados      : 5394
Criterio             : FDR < 0.05 y |log2FC| > 1.0
****************************************************
Inducidos in planta  : 830
Reprimidos in planta : 628
Total diferenciales  : 1458 (27.0 % del genoma filtrado)
